# Ejercicios de Automatización con Python

## Ejercicio 1 

Tu primer script de automatización

In [6]:
# 1. Instalar bibliotecas
!pip install yfinance pandas

In [7]:
# 2. Importar bibliotecas
import yfinance as yf
import pandas as pd

# 3. Definir las acciones a analizar
mis_acciones = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'NVDA']

# 4. Descargar datos históricos de los últimos 6 meses
datos = yf.download(mis_acciones, period='6mo', auto_adjust=True)
precios_cierre = datos['Close']

print("=== Datos descargados ===")
print(precios_cierre.head())

# 5. Estadísticas básicas
print("\n=== Estadísticas básicas (precio de cierre) ===")
print(precios_cierre.describe().round(2))

# 6. Rendimiento diario promedio
rendimientos_diarios = precios_cierre.pct_change().dropna()
print("\n=== Rendimiento diario promedio (%) ===")
print((rendimientos_diarios.mean() * 100).round(4))

# 7. Rendimiento total acumulado
rendimiento_total = (precios_cierre.iloc[-1] / precios_cierre.iloc[0] - 1) * 100
print("\n=== Rendimiento total acumulado en 6 meses (%) ===")
for accion, rend in rendimiento_total.items():
    signo = "▲" if rend > 0 else "▼"
    print(f"  {accion}: {signo} {rend:.2f}%")

# 8. Volatilidad anualizada
volatilidad = rendimientos_diarios.std() * (252 ** 0.5) * 100
print("\n=== Volatilidad anualizada (%) — Mayor valor = Mayor riesgo ===")
print(volatilidad.sort_values(ascending=False).round(2))

# 9. Resumen final
resumen = pd.DataFrame({
    'Precio actual (USD)': precios_cierre.iloc[-1].round(2),
    'Rendimiento 6m (%)': rendimiento_total.round(2),
    'Volatilidad anual (%)': volatilidad.round(2),
    'Rendimiento diario prom (%)': (rendimientos_diarios.mean() * 100).round(4)
})
print("\n=== Resumen comparativo de acciones ===")
print(resumen.sort_values('Rendimiento 6m (%)', ascending=False))

[*********************100%***********************]  5 of 5 completed

=== Datos descargados ===
Ticker            AAPL       GOOGL        MSFT        NVDA        TSLA
Date                                                                  
2025-12-15  273.601654  307.819336  472.714874  176.075226  475.309998
2025-12-16  274.100739  306.171478  474.277924  177.503494  489.880005
2025-12-17  271.335876  296.334259  474.009094  170.731750  467.260010
2025-12-18  271.685242  302.066803  481.834259  173.927856  483.369995
2025-12-19  273.162506  306.760712  483.765656  180.769531  481.200012

=== Estadísticas básicas (precio de cierre) ===
Ticker    AAPL   GOOGL    MSFT    NVDA    TSLA
count   124.00  124.00  124.00  124.00  124.00
mean    271.35  332.69  420.51  192.58  412.27
std      18.09   32.82   34.77   15.28   32.89
min     246.24  273.34  356.00  164.98  343.25
25%     258.61  306.97  397.79  182.55  391.15
50%     268.17  324.46  412.70  187.94  410.87
75%     276.13  351.69  443.89  201.93  433.49
max     315.20  402.38  485.86  235.47  489.88

=== 

---

## Ejercicio 2 — Sistema de alertas básico

Implementa una función que reciba un ticker y un precio objetivo, verifique si el precio actual supera ese objetivo, y retorne un mensaje indicando si se activó la alerta.

In [10]:
import yfinance as yf

def verificar_alerta(ticker, precio_objetivo):
    accion = yf.Ticker(ticker)
    precio_actual = accion.fast_info['last_price']

    if precio_actual >= precio_objetivo:
        return f"ALERTA ACTIVADA — {ticker}: precio actual ${precio_actual:.2f} superó el objetivo de ${precio_objetivo:.2f}"
    else:
        diferencia = precio_objetivo - precio_actual
        return f"Sin alerta — {ticker}: precio actual ${precio_actual:.2f}, faltan ${diferencia:.2f} para alcanzar el objetivo de ${precio_objetivo:.2f}"

# Probar
resultado = verificar_alerta('AAPL', 180)
print(resultado)

ALERTA ACTIVADA — AAPL: precio actual $291.13 superó el objetivo de $180.00


---

## Ejercicio 3 — Análisis de portafolio

Usando la clase `AnalizadorPortafolio`, agrega 2 acciones adicionales, calcula el porcentaje que representa cada acción del total e identifica si el portafolio ganó o perdió en el día.

In [1]:
# Clonar el repo para tener acceso a AnalizadorPortafolio.py
import sys, os

from AnalizadorPortafolio import AnalizadorPortafolio

# Portafolio base + 2 acciones adicionales (META y AMZN)
tickers    = ['AAPL', 'GOOGL', 'MSFT', 'META', 'AMZN']
cantidades = [10,      5,       8,      6,      4]

analizador = AnalizadorPortafolio(tickers, cantidades)
analizador.obtener_datos()

# Calcular porcentaje de cada accion sobre el total
valor_total = analizador.datos['Valor Total'].sum()
analizador.datos['% del total'] = (analizador.datos['Valor Total'] / valor_total * 100).round(2)

print("=== Composicion del portafolio ===")
print(analizador.datos[['Ticker', 'Cantidad', 'Precio', 'Valor Total', '% del total']].to_string(index=False))

# Resultado del dia
ganancia_total = analizador.datos['Ganancia/Perdida $'].sum()
estado = "GANO" if ganancia_total >= 0 else "PERDIO"
signo = "+" if ganancia_total >= 0 else ""
print(f"\nEl portafolio {estado} en el dia: {signo}${ganancia_total:.2f}")

=== Composicion del portafolio ===
Ticker  Cantidad     Precio  Valor Total  % del total
  AAPL        10 291.130005  2911.300049        23.88
 GOOGL         5 359.679993  1798.399963        14.75
  MSFT         8 390.739990  3125.919922        25.64
  META         6 566.979980  3401.879883        27.90
  AMZN         4 238.550003   954.200012         7.83

El portafolio PERDIO en el dia: $-52.79
